# 商场客户聚类

本教程使用客户年龄、年收入和消费积分进行聚类分析。目标不是只得到一个簇标签，而是把聚类结果转化为可解释的客户画像。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print


RANDOM_STATE = 42


def find_project_root(start=None):
    """Find the project root by walking upward to the data directory.

    Args:
        start: Optional start path. Defaults to the current working directory.

    Returns:
        Project root path containing the `data` directory.

    Raises:
        FileNotFoundError: If no parent directory contains `data`.
    """
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("未找到包含 data/ 的项目根目录")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler


## 1. 读取与标准化字段


In [ ]:
raw_customers = pd.read_csv(DATA_DIR / "mall_customers.csv", dtype={"CustomerID": str})
customers = (
    raw_customers.rename(
        columns={
            "CustomerID": "customer_id",
            "Gender": "gender",
            "Age": "age",
            "Annual Income (k$)": "annual_income",
            "Spending Score (1-100)": "spending_score",
        }
    )
    .set_index("customer_id")
    .sort_index()
)

display(customers.head())
display(customers.isna().sum().to_frame("missing_count"))


## 2. 缺失值与异常值处理


In [ ]:
def cap_outliers_iqr(frame, columns, whisker=1.5):
    """Cap numeric outliers with the IQR rule.

    Args:
        frame: Source data frame.
        columns: Numeric columns to process.
        whisker: IQR multiplier used to compute lower and upper bounds.

    Returns:
        A tuple containing the capped data frame and a bounds report.
    """
    result = frame.copy()
    bounds = []
    for column in columns:
        q1 = result[column].quantile(0.25)
        q3 = result[column].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - whisker * iqr
        upper = q3 + whisker * iqr
        result[column] = result[column].clip(lower=lower, upper=upper)
        bounds.append({"feature": column, "lower": lower, "upper": upper})
    return result, pd.DataFrame(bounds).set_index("feature")


numeric_columns = ["age", "annual_income", "spending_score"]
customers[numeric_columns] = customers[numeric_columns].apply(pd.to_numeric, errors="coerce")
customers["annual_income"] = customers["annual_income"].fillna(customers["annual_income"].median())

cleaned_customers, outlier_bounds = cap_outliers_iqr(
    customers, columns=["annual_income", "spending_score"]
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=120)
customers[["annual_income", "spending_score"]].boxplot(ax=axes[0])
cleaned_customers[["annual_income", "spending_score"]].boxplot(ax=axes[1])
axes[0].set_title("处理前")
axes[1].set_title("IQR 截尾后")
for ax in axes:
    ax.set_ylabel("取值")
plt.tight_layout()
plt.show()

display(outlier_bounds.round(2))
display(cleaned_customers.describe().round(2))


## 3. 年收入与消费积分分布


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
ax.scatter(
    cleaned_customers["annual_income"],
    cleaned_customers["spending_score"],
    alpha=0.8,
)
ax.set_title("年收入与消费积分分布")
ax.set_xlabel("年收入 (k$)")
ax.set_ylabel("消费积分 (1-100)")
ax.grid(alpha=0.25)
plt.show()


## 4. 选择 K-Means 的聚类数量


In [ ]:
features = cleaned_customers[["annual_income", "spending_score"]]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

k_records = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels = model.fit_predict(scaled_features)
    k_records.append(
        {
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette_score(scaled_features, labels),
            "calinski_harabasz": calinski_harabasz_score(scaled_features, labels),
        }
    )

k_report = pd.DataFrame(k_records)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
axes[0].plot(k_report["k"], k_report["inertia"], marker="o")
axes[0].set_title("肘部法")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[1].plot(k_report["k"], k_report["silhouette"], marker="o", color="darkorange")
axes[1].set_title("轮廓系数")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

display(k_report.round(4))


## 5. 训练 K-Means 并解释客户群


In [ ]:
kmeans = KMeans(n_clusters=5, random_state=RANDOM_STATE, n_init=20)
customers_with_cluster = cleaned_customers.copy()
customers_with_cluster["cluster"] = kmeans.fit_predict(scaled_features)

centers = pd.DataFrame(
    scaler.inverse_transform(kmeans.cluster_centers_),
    columns=["annual_income", "spending_score"],
)
centers.index.name = "cluster"

fig, ax = plt.subplots(figsize=(7, 5), dpi=120)
scatter = ax.scatter(
    customers_with_cluster["annual_income"],
    customers_with_cluster["spending_score"],
    c=customers_with_cluster["cluster"],
    cmap="tab10",
    alpha=0.85,
)
ax.scatter(centers["annual_income"], centers["spending_score"], marker="*", s=260, c="black")
ax.set_title("K-Means 客户聚类")
ax.set_xlabel("年收入 (k$)")
ax.set_ylabel("消费积分 (1-100)")
ax.grid(alpha=0.25)
ax.legend(*scatter.legend_elements(), title="cluster", loc="best")
plt.show()

profile = customers_with_cluster.groupby("cluster").agg(
    customer_count=("age", "size"),
    avg_age=("age", "mean"),
    avg_income=("annual_income", "mean"),
    avg_score=("spending_score", "mean"),
    female_rate=("gender", lambda s: (s == "Female").mean()),
)

display(centers.round(2))
display(profile.round(2))


## 6. 年龄与性别消费分析


In [ ]:
age_score = customers_with_cluster.groupby("age")["spending_score"].sum()
gender_score = customers_with_cluster.groupby("gender")["spending_score"].sum()

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)
age_score.plot(kind="bar", ax=axes[0], width=0.8)
axes[0].set_title("各年龄消费积分总和")
axes[0].set_xlabel("年龄")
axes[0].set_ylabel("消费积分总和")

gender_score.plot(kind="pie", autopct="%.1f%%", ax=axes[1])
axes[1].set_title("不同性别消费积分占比")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()


## 7. 对比其他聚类算法


In [ ]:
def evaluate_clustering(name, features, labels):
    """Evaluate a clustering result with common unsupervised metrics.

    Args:
        name: Display name of the clustering model.
        features: Feature matrix used by the clustering model.
        labels: Predicted cluster labels. DBSCAN noise can be marked as -1.

    Returns:
        Dictionary containing cluster count and evaluation metrics.
    """
    unique_labels = set(labels)
    if len(unique_labels) < 2:
        return {
            "model": name,
            "clusters": len(unique_labels),
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
        }
    return {
        "model": name,
        "clusters": len(unique_labels - {-1}),
        "silhouette": silhouette_score(features, labels),
        "calinski_harabasz": calinski_harabasz_score(features, labels),
        "davies_bouldin": davies_bouldin_score(features, labels),
    }


dbscan = DBSCAN(eps=0.45, min_samples=5)
agglomerative = AgglomerativeClustering(n_clusters=5)

model_labels = {
    "K-Means": customers_with_cluster["cluster"].to_numpy(),
    "DBSCAN": dbscan.fit_predict(scaled_features),
    "Agglomerative": agglomerative.fit_predict(scaled_features),
}

evaluation = pd.DataFrame(
    evaluate_clustering(name, scaled_features, labels)
    for name, labels in model_labels.items()
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=120, sharex=True, sharey=True)
for ax, (name, labels) in zip(axes, model_labels.items()):
    ax.scatter(features["annual_income"], features["spending_score"], c=labels, cmap="tab10", alpha=0.85)
    ax.set_title(name)
    ax.set_xlabel("年收入 (k$)")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("消费积分 (1-100)")
plt.tight_layout()
plt.show()

display(evaluation.round(4))
